# TP4 — Aller plus loin : trois techniques pour mieux chercher

Vous avez terminé les trois TP ? Ce notebook est pour vous.

Au TP3, on a appris à **mesurer** une recherche : un jeu de test, le hit@k, le MRR. Il reste des questions dont la
bonne page n'arrive pas en tête. On teste ici trois techniques courantes dans les RAG en production.
Chacune agit à un endroit différent de la chaîne :

```
indexation :  chunks ──► [3. enrichissement] ──► embeddings ──► index
question   :  question ──► [2. HyDE] ──► recherche (n candidats) ──► [1. reranking] ──► k chunks ──► LLM
```

Pour chaque technique, la démarche est celle du TP3 :

1. comprendre l'idée et le problème qu'elle corrige ;
2. compléter une fonction (repérée par `...`) ;
3. mesurer la variante sur le jeu de test et l'ajouter au tableau `resultats` ;
4. analyser : qu'est-ce qui gagne, qu'est-ce qui perd, et à quel coût ?

Plan :
0. Le point de départ
1. Le reranking : reclasser les candidats (sans clé d'API, environ 5 min)
2. HyDE : chercher avec une réponse imaginée (une cinquantaine d'appels au LLM)
3. Enrichir les chunks avant l'indexation (environ 600 appels au LLM, 5 à 10 min)
4. Bilan : combiner et choisir

Les parties 1, 2 et 3 sont indépendantes. Le bilan reprend les fonctions des parties 1 et 3.

**Prérequis** : les corrigés `solutions/tp1.py`, `solutions/tp2.py` et `solutions/tp3.py`. On y importe la recherche
hybride (TP1), la réécriture de question (TP2) et les fonctions d'évaluation (TP3).

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
from fastembed.rerank.cross_encoder import TextCrossEncoder

import settings
from rag import load_index, load_pdfs, save_index, embed, search, generate
from solutions.tp1 import search_hybrid
from solutions.tp2 import rewrite_query
from solutions.tp3 import evaluer_retrieval, hit_at_k, mrr, afficher_resultats

chunks, embeddings = load_index()
with open("eval/questions.json", encoding="utf-8") as f:
    jeu_test = json.load(f)
print(len(jeu_test), "questions,", len(chunks), "chunks")

## 0. Le point de départ

On reprend deux variantes du TP3 comme références : la recherche sémantique seule et la recherche hybride (RRF).
Chaque technique de ce TP devra faire mieux que l'une ou l'autre.

`afficher_resultats` affiche maintenant le MRR à côté du hit@k.
Rappel : le jeu compte 26 questions, donc **une question pèse environ 0,04**. Un écart de 0,04, c'est une seule
question : ne sur-interprétez pas les petits écarts.

In [ ]:
resultats = {
    "sémantique": evaluer_retrieval(lambda q, k: search(q, chunks, embeddings, k), jeu_test),
    "hybride (RRF)": evaluer_retrieval(lambda q, k: search_hybrid(q, chunks, embeddings, k), jeu_test),
}
afficher_resultats(resultats)

## 1. Le reranking : reclasser les candidats

**Le problème.** La recherche sémantique compare deux vecteurs calculés **séparément** : celui de la question et
celui du chunk. Le vecteur du chunk est calculé une fois pour toutes, sans savoir quelle question on posera.
Il résume tout le chunk en 384 nombres. C'est très rapide (un produit scalaire), mais approximatif : des détails
se perdent.

**L'idée.** Un **cross-encoder** lit la question et le chunk **ensemble**, en une seule passe du modèle, et renvoie
un score de pertinence. Il voit quel mot de la question répond à quel mot du chunk. C'est plus précis.
En contrepartie, il faut relancer le modèle pour chaque paire (question, chunk) : impossible sur les 601 chunks
à chaque question.

**La solution : deux étages.**

1. La recherche rapide ramène `n` candidats (ici 20) : on jette un filet large.
2. Le cross-encoder reclasse ces `n` candidats, et on garde les `k` meilleurs.

C'est le principe d'un recrutement : un premier tri rapide sur les CV, puis un entretien pour les vingt retenus.

**Le modèle** : `settings.RERANK_MODEL` (`Xenova/ms-marco-MiniLM-L-6-v2`), 80 Mo, téléchargé au premier appel.
Il a été entraîné sur MS MARCO, un jeu de requêtes du moteur Bing **en anglais**. Gardez ce détail en tête.

La cellule suivante charge le modèle et lui fait noter trois textes, pour la même question posée en anglais puis
en français.

In [ ]:
reranker = TextCrossEncoder(settings.RERANK_MODEL)

textes = [
    "Working Voltage: DC 5V. Working Current: 15mA. Ranging distance: 2cm - 400cm.",
    "The LM35 device is rated to operate over a -55 to 150 C temperature range.",
    "ESP32 supports Wi-Fi 802.11 b/g/n and Bluetooth v4.2 BR/EDR and BLE.",
]
for question in ["What is the supply voltage of the HC-SR04?", "Quelle est la tension d'alimentation du HC-SR04 ?"]:
    scores = list(reranker.rerank(question, textes))  # un score par texte, dans le même ordre
    print(question)
    for score, texte in zip(scores, textes):
        print(f"   {score:6.2f}  {texte[:60]}")

**À observer**

- Le score n'est pas une probabilité : seul l'ordre compte. Plus il est haut, plus le texte est jugé pertinent.
- En anglais, le bon texte se détache-t-il ? Et en français ? Comparez les écarts entre les trois scores.

### Exercice 1 — la fonction `rerank`

Complétez `rerank(question, candidats, k)`. Elle reçoit une liste de chunks (les candidats), les note avec
`reranker.rerank`, puis renvoie les `k` meilleurs, du plus pertinent au moins pertinent.
Gardez le format de `search` : chaque chunk est un dictionnaire, avec son score dans le champ `"score"`.

Indice : `np.argsort(scores)[::-1][:k]` donne les indices des `k` plus grands scores, comme dans `rag.search`.

In [ ]:
def rerank(question, candidats, k=5):
    """Reclasse les candidats avec le cross-encoder et garde les k meilleurs."""
    ...  # à compléter


def search_rerank(question, chunks, embeddings, k=5, n=20):
    """Deux étages : la recherche sémantique ramène n candidats, le cross-encoder en garde k."""
    return rerank(question, search(question, chunks, embeddings, k=n), k)

**Mesure.** On mesure le reranking avec 20 puis 50 candidats, et on chronomètre : le cross-encoder tourne sur
votre processeur. Comptez 1 à 3 minutes par variante.

In [ ]:
for n in [20, 50]:
    debut = time.perf_counter()
    resultats[f"rerank (n={n})"] = evaluer_retrieval(lambda q, k: search_rerank(q, chunks, embeddings, k, n=n), jeu_test)
    print(f"n={n} : {(time.perf_counter() - debut) / len(jeu_test):.1f} s par question")
afficher_resultats(resultats)

**Le plafond du premier étage.** Le reranker ne fait que reclasser : il ne peut pas trouver une page que la
recherche n'a pas ramenée. Combien de bonnes pages se trouvent parmi les `n` candidats ?

In [ ]:
for n in [20, 50]:
    rangs_n = evaluer_retrieval(lambda q, k: search(q, chunks, embeddings, k), jeu_test, k_max=n)
    print(f"bonne page parmi les {n} candidats : {hit_at_k(rangs_n, n):.2f}")

**À analyser**

- Comparez le hit@5 du reranking au plafond de son premier étage. Que reste-t-il à gagner en reclassant mieux ?
  Et en élargissant `n` ?
- Le hit@5 et le MRR progressent-ils autant l'un que l'autre ? Repensez à l'expérience en français : le reranker
  est-il aussi bon pour mettre la bonne page **en tête** que pour la faire entrer dans le top 5 ?
- Quel est le prix ? Comparez le temps par question avec celui de la recherche seule (environ 0,1 s).
- Pistes : reclasser les candidats de la recherche **hybride** ; donner au reranker la question en anglais
  (`rewrite_query` du TP2) ; essayer un reranker multilingue (`jinaai/jina-reranker-v2-base-multilingual`,
  1,1 Go à télécharger). Le bilan teste les deux premières.

## 2. HyDE : chercher avec une réponse imaginée

**Le problème.** Comparez une question du jeu de test et le passage qui y répond :

- question : « Quelle est la tension d'alimentation du HC-SR04 ? »
- passage : « Working Voltage: DC 5V. Working Current: 15mA. Ranging distance: 2cm - 400cm. »

La langue diffère, la forme aussi : une question d'un côté, des lignes de tableau de l'autre. Leurs vecteurs ne sont
pas si proches. Le vecteur d'une question ressemble souvent plus à celui d'une autre question qu'à celui de sa réponse.

**L'idée.** HyDE (*Hypothetical Document Embeddings*, Gao et al., 2022). On demande au LLM d'écrire le passage qu'il
**imagine** trouver dans la datasheet. Puis on cherche avec ce passage au lieu de la question.
On compare alors une réponse à des réponses.

**Et si le LLM invente une valeur fausse ?** Ce n'est pas grave ici. Le passage imaginé sert seulement à **trouver**
les chunks ; la réponse finale vient toujours des vrais extraits. Une valeur fausse avec le bon vocabulaire
(« Working Voltage », « DC », « V ») suffit à orienter la recherche.

**Le lien avec le TP2.** L'exercice 3 du TP2 réécrivait la question en une requête courte, en anglais. HyDE va plus
loin : il ne reformule pas la question, il imagine la réponse. On compare les deux.

**Le coût** : un appel au LLM par question, avant même la recherche. Cette partie fait 3 + 26 + 26 = 55 appels.

### Exercice 2 — la fonction `document_hypothetique`

Complétez le prompt et la fonction. Le prompt demande un passage court, en anglais, dans le style d'une datasheet,
avec des valeurs et des unités. La fonction appelle le LLM avec `generate` (de `rag.py`), qui prend une liste de
messages et renvoie le texte de la réponse.

In [ ]:
PROMPT_HYDE = """..."""  # à compléter : placez {question} là où doit apparaître la question


def document_hypothetique(question):
    """Demande au LLM le passage qu'il imagine trouver dans la datasheet."""
    ...  # à compléter


def search_hyde(question, chunks, embeddings, k=5):
    """Cherche avec le passage hypothétique au lieu de la question."""
    return search(document_hypothetique(question), chunks, embeddings, k)

Testez sur trois questions avant de lancer la mesure :

In [ ]:
for item in jeu_test[:3]:
    print("question :", item["question"], f"  (réponse : {item['source']} p.{item['pages']})")
    print(document_hypothetique(item["question"]), "\n")

Lisez les passages. Le vocabulaire ressemble-t-il à celui d'une datasheet ? Les valeurs sont-elles justes ?
Ouvrez le PDF à la page indiquée pour vérifier.

**Mesure** : HyDE contre la réécriture du TP2 (26 appels chacune).

In [ ]:
resultats["HyDE"] = evaluer_retrieval(lambda q, k: search_hyde(q, chunks, embeddings, k), jeu_test)
resultats["réécriture (TP2)"] = evaluer_retrieval(lambda q, k: search(rewrite_query(q), chunks, embeddings, k), jeu_test)
afficher_resultats(resultats)

# rang de la bonne page, question par question : sémantique → HyDE
for item, rang_base, rang_hyde in zip(jeu_test, resultats["sémantique"], resultats["HyDE"]):
    print(f"{rang_base or '-':>2} → {rang_hyde or '-':>2}  {item['question']}")

**À analyser**

- HyDE fait-il mieux que la question brute ? Que la réécriture du TP2 ?
- Question par question : où HyDE gagne-t-il, où perd-il ? Indice : que se passe-t-il quand le LLM imagine une
  datasheet différente de la vraie (autre vocabulaire, autre présentation, valeur très éloignée) ?
- Relancez la mesure : obtenez-vous exactement les mêmes chiffres ? Qu'en concluez-vous sur une variante qui
  dépend d'un LLM ?
- Le coût : un appel au LLM par question, soit une à deux secondes de plus avant chaque réponse. Le gain le
  justifie-t-il ?
- Pistes : chercher avec la question **et** le passage mis bout à bout ; utiliser le passage dans la recherche
  hybride (attention : BM25 cherchera aussi les valeurs inventées).

## 3. Enrichir les chunks avant l'indexation

**Le problème.** Le découpage coupe tous les 150 mots, sans regarder le sens. Isolé, un chunk perd son contexte.
Exemple réel, un chunk de la page 19 de la datasheet du BME280 :

> « … and performance for weather monitoring Suggested settings for weather monitoring Sensor mode forced mode,
> 1 sample / minute … Current consumption 0.16 µA … »

Il ne nomme jamais le composant. Comment la recherche saurait-elle qu'il parle du BME280 ? Dans l'index de référence,
98 chunks de plus de 100 mots ne nomment pas leur composant.

**L'idée.** Avant de calculer les embeddings, on ajoute en tête de chaque chunk une phrase qui le situe : le composant,
la partie du document, le sujet. Par exemple : « Extrait de la datasheet du capteur BME280, réglages conseillés et
consommation pour une station météo. » Le chunk se comprend alors seul. On parle souvent de *contextual retrieval*
(recherche contextuelle).

**Le lien avec le TP1.** Le découpage par paragraphes ajoutait déjà le préfixe `[BME280]`. C'est un enrichissement
minimal, sans LLM. On mesure d'abord ce préfixe seul, puis la phrase rédigée par le LLM : on saura ce que le LLM
apporte en plus.

**Le coût.** Un appel au LLM par chunk, soit environ 600 appels. Mais on les paie **une seule fois**, à
l'indexation, et non à chaque question comme HyDE. On ne les relance que si les documents changent.

### Étape 1 — le préfixe seul, sans LLM

Calcul des embeddings : 2 à 5 minutes selon votre machine.

In [ ]:
chunks_prefixe = [{**c, "text": f"[{c['source'].removesuffix('.pdf').upper()}] {c['text']}"} for c in chunks]
embeddings_prefixe = embed([c["text"] for c in chunks_prefixe])

resultats["préfixe [COMPOSANT]"] = evaluer_retrieval(lambda q, k: search(q, chunks_prefixe, embeddings_prefixe, k), jeu_test)
afficher_resultats(resultats)

### Exercice 3 — la phrase de contexte

Complétez `PROMPT_CONTEXTE` et `contexte_chunk(chunk, texte_page)`.
Le LLM reçoit la page entière, pour comprendre le contexte, et le chunk, qu'il doit situer. Il répond par une seule
phrase.

Conseils :

- Coupez la page à 3000 caractères pour limiter les tokens.
- Phrase en français ou en anglais ? Les questions sont en français, les datasheets en anglais. À vous de choisir…
  et de mesurer.

La cellule teste votre fonction sur trois chunks. Vérifiez les phrases avant de lancer les 600 appels.

In [ ]:
PROMPT_CONTEXTE = """..."""  # à compléter : placez {source}, {page} et {extrait} dans le texte


def contexte_chunk(chunk, texte_page):
    """Une phrase de contexte rédigée par le LLM, qui situe le chunk dans son document."""
    ...  # à compléter


pages = load_pdfs()
texte_page = {(p["source"], p["page"]): p["text"] for p in pages}
for chunk in [chunks[84], chunks[109], chunks[169]]:
    print(chunk["source"], "p.", chunk["page"])
    print("   contexte :", contexte_chunk(chunk, texte_page[(chunk["source"], chunk["page"])]))
    print("   extrait  :", chunk["text"][:150], "\n")

### Étape 2 — enrichir tout l'index

La cellule suivante est complète. Elle appelle votre `contexte_chunk` sur les 601 chunks, huit appels à la fois
(avec `ThreadPoolExecutor`, de la bibliothèque standard), puis calcule les embeddings et sauvegarde le résultat dans `index_enrichi/`. Comptez 5 à 10 minutes.

Si `index_enrichi/` existe déjà, la cellule le recharge. Supprimez ce dossier pour tout recalculer, par exemple
après avoir changé le prompt.

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def enrichir_chunks(chunks, pages, fonction_contexte, appels_simultanes=8):
    """Ajoute à chaque chunk sa phrase de contexte, en tête du texte.

    Les appels partent huit par huit : l'un après l'autre, 600 appels prendraient 10 à 20 minutes.
    """
    texte_page = {(p["source"], p["page"]): p["text"] for p in pages}

    def enrichir(chunk):
        try:
            contexte = fonction_contexte(chunk, texte_page[(chunk["source"], chunk["page"])])
        except Exception as erreur:  # un appel en échec ne doit pas faire perdre les 600 autres
            print(f"chunk {chunk['id']} : appel en échec ({type(erreur).__name__}), chunk gardé sans contexte")
            contexte = ""
        return {**chunk, "contexte": contexte, "text": f"{contexte}\n{chunk['text']}".strip()}

    with ThreadPoolExecutor(max_workers=appels_simultanes) as executor:
        chunks_enrichis = list(executor.map(enrichir, chunks))
    sans_contexte = sum(not c["contexte"] for c in chunks_enrichis)
    if sans_contexte:
        print(f"{sans_contexte} chunks sur {len(chunks)} sans contexte : vérifiez la clé d'API et relancez.")
    return chunks_enrichis


if Path("index_enrichi").exists():
    chunks_enrichis, embeddings_enrichis = load_index("index_enrichi")
else:
    debut = time.perf_counter()
    chunks_enrichis = enrichir_chunks(chunks, pages, fonction_contexte=contexte_chunk)
    embeddings_enrichis = embed([c["text"] for c in chunks_enrichis])
    save_index(chunks_enrichis, embeddings_enrichis, "index_enrichi")
    print(f"index enrichi en {(time.perf_counter() - debut) / 60:.1f} min")

resultats["contexte LLM"] = evaluer_retrieval(
    lambda q, k: search(q, chunks_enrichis, embeddings_enrichis, k), jeu_test)
resultats["contexte LLM + hybride"] = evaluer_retrieval(
    lambda q, k: search_hybrid(q, chunks_enrichis, embeddings_enrichis, k), jeu_test)
afficher_resultats(resultats)

**À analyser**

- Le préfixe seul apporte-t-il déjà l'essentiel ? Que gagne-t-on en plus avec la phrase du LLM ?
- Lisez quelques contextes (`chunks_enrichis[i]["contexte"]`). Sont-ils justes ? Un contexte faux peut-il égarer la
  recherche ?
- La recherche hybride profite-t-elle aussi de l'enrichissement ? Indice : BM25 compte les mots, et la phrase
  ajoute des mots.
- Le coût : 600 appels avec la page à chaque fois, combien de tokens ? Que se passe-t-il quand un document change ?
  Et pour un corpus de 10 000 pages ?
- Piste : rédiger le contexte une fois par **page** plutôt que par chunk (236 appels au lieu de 601).

## 4. Bilan : combiner et choisir

Les trois techniques agissent à des endroits différents : on peut les cumuler. Mais les gains s'additionnent-ils ?
La cellule mesure trois combinaisons autour de la meilleure recherche du TP1, l'hybride :

- hybride + reranking ;
- hybride + reranking, avec la question réécrite en anglais pour le reranker (26 appels au LLM) ;
- index enrichi + hybride + reranking, si vous avez fait la partie 3.

Comptez 3 à 5 minutes.

In [ ]:
resultats["hybride + rerank"] = evaluer_retrieval(
    lambda q, k: rerank(q, search_hybrid(q, chunks, embeddings, 20), k), jeu_test)

# la recherche se fait avec la question d'origine ; seul le reranker reçoit la question en anglais
resultats["hybride + rerank (anglais)"] = evaluer_retrieval(
    lambda q, k: rerank(rewrite_query(q), search_hybrid(q, chunks, embeddings, 20), k), jeu_test)

if "chunks_enrichis" in globals():
    resultats["enrichi + hybride + rerank"] = evaluer_retrieval(
        lambda q, k: rerank(q, search_hybrid(q, chunks_enrichis, embeddings_enrichis, 20), k), jeu_test)

afficher_resultats(resultats)

**Le coût de chaque technique**

| technique | où elle agit | ce qu'elle coûte | quand on le paie |
|---|---|---|---|
| reranking | après la recherche | une passe du cross-encoder par candidat, sur le processeur | à chaque question |
| HyDE | avant la recherche | un appel au LLM | à chaque question |
| enrichissement | à l'indexation | un appel au LLM par chunk | une fois, puis à chaque mise à jour des documents |

**À conclure**

- Quelle combinaison retiendriez-vous pour ce corpus ? Justifiez avec vos chiffres **et** avec les coûts.
- Les gains se sont-ils additionnés ? Une brique de plus n'est pas toujours un gain : seule la mesure le dit.
- 26 questions, c'est peu. Avant de décider pour de bon, que faudrait-il ajouter au jeu de test ?

```
mesurer (TP3)  ──►  une idée, une variante  ──►  mesurer à nouveau  ──►  garder ce qui gagne, au juste coût
```

C'est la boucle de travail sur un RAG en production : chaque technique est une hypothèse, et le jeu de test tranche.